In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv('../data/processed/punjab_district_weekly_merged.csv')

def encyclicEncoding(data):

    data["month_sin"] = np.sin(2 * np.pi * data["month_num"] / 12)
    data["month_cos"] = np.cos(2 * np.pi * data["month_num"] / 12)
    data["week_sin"] = np.sin(2 * np.pi * data["epi_week"] / 52)
    data["week_cos"] = np.cos(2 * np.pi * data["epi_week"] / 52)

    return data

data=encyclicEncoding(data)
data.to_csv("../data/processed/updated-Punjab.csv", index=False)



In [2]:
import pandas as pd
import numpy as np

data = pd.read_csv('../data/processed/updated-Punjab.csv')


def HandleDataInconsistency(data):
        
    data["district"] = data["district"].ffill()
    data["month"] = data["month"].ffill()

    data["cases"] = data["cases"].interpolate(method="linear")
    data["deaths"] = data["deaths"].interpolate(method="linear")


    data = data.drop_duplicates()

    data["district"] = data["district"].str.lower().str.strip()


    data["cases"] = data["cases"].clip(lower=0)
    data["deaths"] = data["deaths"].clip(lower=0)


    data = data[(data["epi_week"] >= 1) & (data["epi_week"] <= 52)]
    data = data[(data["month_num"] >= 1) & (data["month_num"] <= 12)]


    data = data.sort_values(["district", "year", "epi_week"])


    assert data["month_num"].between(1, 12).all()
    assert data["epi_week"].between(1, 52).all()

    return data

data=HandleDataInconsistency(data)
data.to_csv("../data/processed/updated-Punjab.csv", index=False)

In [3]:
import numpy as np
import pandas as pd

def lag_features(df):

    df=df.sort_values(by=["district","year", "epi_week"]).reset_index(drop=True)

    df["t1_cases"]=df.groupby("district")["cases"].shift(1)
    df["t2_cases"]=df.groupby("district")["cases"].shift(2)
    
    df["T2M_lag1"]=df.groupby("district")["T2M_mean"].shift(1)

    df["PRECTOTCORR_lag1"]=df.groupby("district")["PRECTOTCORR_sum"].shift(1)
    df["PRECTOTCORR_lag2"]=df.groupby("district")["PRECTOTCORR_sum"].shift(2)

    df=df.dropna(subset=["t1_cases", "t2_cases"]).reset_index(drop=True)
    return df

data = pd.read_csv('../data/processed/updated-Punjab.csv')
data=lag_features(data)
data.to_csv("../data/processed/updated-Punjab.csv", index=False)

In [4]:
def water_proxy(df):

    df=df.sort_values(by=["district","year", "epi_week"]).reset_index(drop=True)

    df["water_proxy"]=(df.groupby("district")["PRECTOTCORR_sum"]
                        .transform(lambda elem:elem.rolling(window=2,min_periods=1).sum()))
    
    return df

data = pd.read_csv('../data/processed/updated-Punjab.csv')
data=water_proxy(data)
data.to_csv("../data/processed/updated-Punjab.csv", index=False)

In [5]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
import joblib

df = pd.read_csv('../data/processed/updated-Punjab.csv')

# Sort
df = df.sort_values(["district", "year", "epi_week"]).reset_index(drop=True)

# Train mask — defined once, used everywhere
train_mask = df['year'].isin([2016, 2017, 2018])

# Encode district on train only
encoder = OrdinalEncoder()
encoder.fit(df[train_mask][['district']])
df['district_encoded'] = encoder.transform(df[['district']])
joblib.dump(encoder, "../models/district_encoder.pkl")

# Outbreak threshold from train only
district_threshold = df[train_mask].groupby('district')['cases'].quantile(0.90)
df['districtThreshold'] = df['district'].map(district_threshold) 
df['isOutbreak'] = (df['cases'] > df['districtThreshold']).astype(int)
df = df.drop(columns=['districtThreshold'])  # cleanup helper column

# Year-wise split
train = df[df['year'].isin([2016, 2017, 2018])].copy()
val   = df[df['year'] == 2019].copy()
test  = df[df['year'] == 2020].copy()

n = len(df)
print(f"Total rows : {n}")
print(f"Train rows : {len(train)} ({len(train)/n*100:.1f}%) — 2016-2018")
print(f"Val rows   : {len(val)}   ({len(val)/n*100:.1f}%) — 2019 outbreak")
print(f"Test rows  : {len(test)}  ({len(test)/n*100:.1f}%) — 2020")
print(f"\nOutbreak weeks in train : {train['isOutbreak'].sum()}")
print(f"Outbreak weeks in val   : {val['isOutbreak'].sum()}")
print(f"Outbreak weeks in test  : {test['isOutbreak'].sum()}")

Total rows : 8568
Train rows : 5112 (59.7%) — 2016-2018
Val rows   : 1728   (20.2%) — 2019 outbreak
Test rows  : 1728  (20.2%) — 2020

Outbreak weeks in train : 454
Outbreak weeks in val   : 647
Outbreak weeks in test  : 109


In [6]:
#Feature Engineering 
#in our case,the features are mostly the derived and engineered rows like the lag features,water proxy,encycling-features and the weather condition while our target is the no. of cases

Features = [ 
"district_encoded",    # Encoded district (categorical)
"T2M_mean",          # Current temperature  
"RH2M_mean",          # Current humidity 
"PRECTOTCORR_sum",    # Current precipitation 
"WS10M_mean",        # Current wind speed 
"T2M_lag1",          # Previous week temperature (delayed effect) 
"PRECTOTCORR_lag1",   # Previous week rainfall 
"PRECTOTCORR_lag2",   # Two-week rainfall (standing water) 
"t1_cases",           # Cases last week (autoregressive) 
"t2_cases",           # Cases two weeks ago 
"water_proxy",        
"month_sin",          
"month_cos",          
"week_sin",          
"week_cos",
"isOutbreak"          
]

Target = "cases"  # Number of dengue cases to predict

xtrain = train[Features].copy()
ytrain = train[Target].copy()
xval = val[Features].copy()
yval = val[Target].copy()
xtest = test[Features].copy()
ytest = test[Target].copy()

print(f"Feature matrix shape: {xtrain.shape}") 
print(f"Target vector shape:  {ytrain.shape}")

Feature matrix shape: (5112, 16)
Target vector shape:  (5112,)


In [7]:
# Check before modeling
print(xtrain.isnull().sum())

district_encoded    0
T2M_mean            0
RH2M_mean           0
PRECTOTCORR_sum     0
WS10M_mean          0
T2M_lag1            0
PRECTOTCORR_lag1    0
PRECTOTCORR_lag2    0
t1_cases            0
t2_cases            0
water_proxy         0
month_sin           0
month_cos           0
week_sin            0
week_cos            0
isOutbreak          0
dtype: int64


In [8]:
#Scaling the data
from sklearn.preprocessing import StandardScaler
import joblib as jl

scaler = StandardScaler()
xtrain_scaled = scaler.fit_transform(xtrain)
xval_scaled = scaler.transform(xval)
xtest_scaled = scaler.transform(xtest)
jl.dump(scaler, "../models/scaler.pkl")

print(f"Scaler means (first 5 features): {scaler.mean_[:5]}") 
print(f"Scaler stds  (first 5 features): {scaler.scale_[:5]}")

Scaler means (first 5 features): [17.5        26.71619495 36.0387173   8.2737813   2.52306729]
Scaler stds  (first 5 features): [10.38829469  8.30447763 13.64367141 15.44411543  0.72919974]


In [9]:

severityMapping={
    0:'Low',
    1:'Medium',
    2:'High',
    3:'Critical'
}

q25=ytrain.quantile(0.25)
q75=ytrain.quantile(0.75)
q90=ytrain.quantile(0.90)

def labelSeverityCases(cases):
    if cases <= q25:
        return 0  # Low
    elif cases <= q75:
        return 1  # Medium
    elif cases <= q90:
        return 2  # High
    else:
        return 3  # Critical
    

yTrainseverity = ytrain.apply(labelSeverityCases).values
yvalseverity = yval.apply(labelSeverityCases).values                                                   
ytestseverity = ytest.apply(labelSeverityCases).values



In [11]:
# freezing the data pipeline for saving the data from exposure to risks and corruption
import numpy as np
train.to_csv("../data/processed/train.csv", index=False)
val.to_csv("../data/processed/val.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

np.save("../data/processed/xtrain_scaled.npy", xtrain_scaled)
np.save("../data/processed/ytrain_severity.npy", yTrainseverity)